In [5]:
import sys
from pathlib import Path

# project/
project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

In [6]:
from collections import defaultdict
from pathlib import Path
import pickle

from rank_bm25 import BM25Okapi
from week5_ragfoundations.day4.retrival import retrieve
from sentence_transformers import CrossEncoder

from dotenv import load_dotenv
import os
import json
from groq import Groq


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3288.92it/s]


In [7]:
#bm25 retrival
BASE_DIR = Path.cwd()

METADATA_PATH = (
    BASE_DIR.parent.parent
    / "week5_ragfoundations"
    / "day4"
    / "metadata.pkl"
)
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

# print(metadata)

documents = [
    item["text"]
    for item in metadata
]


# print(documents)


tokenized_docs = [
    doc.lower().split()
    for doc in documents
]


bm25 = BM25Okapi(tokenized_docs)

# QUERY = input("Enter your query: ")

def bm25_retrieve(query, top_k=20):
    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "chunk_id": metadata[idx]["chunk_id"],
            "text": metadata[idx]["text"],
            "source": metadata[idx]["source"],
            "rank": rank
        })

    return results

# bm25_results = bm25_retrieve(QUERY, top_k=20)
# bm25_results

In [8]:
#RRF reciprocal rank fusion

def reciprocal_rank_fusion(result_lists, k=60):
    scores = defaultdict(float)
    chunks = {}

    for results in result_lists:

        for rank, chunk in enumerate(results):

            chunk_key = (chunk["source"], chunk["chunk_id"])

            scores[chunk_key] += 1 / (k + rank + 1)

            chunks[chunk_key] = chunk

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    fused_results = []

    for chunk_key, _ in ranked:
        fused_results.append(chunks[chunk_key])

    return fused_results

In [9]:
def hybrid_retrieve(query, top_k=5, dense_top_k=20, bm25_top_k=20):

    dense_results = retrieve(query, top_k=dense_top_k)

    bm25_results = bm25_retrieve(query, top_k=bm25_top_k)

    final_results = reciprocal_rank_fusion([
        dense_results,
        bm25_results
    ])

    return final_results[:top_k]

In [10]:
# dense_results = retrieve(QUERY, top_k=20)
# bm25_results = bm25_retrieve(QUERY, top_k=20)

# final_results = reciprocal_rank_fusion(
#     [dense_results, bm25_results]
# )

# dense_results
# final_results

In [11]:
#iske pehle jo bhi tha sab bi-enocder tha

#cross encoder
model = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3978.48it/s]


In [12]:
def crossencoding(query, retrieved_chunks=None, top_k=5):
    if retrieved_chunks is None:
        retrieved_chunks = hybrid_retrieve(query, top_k=20)

    pairs = [(query, i['text']) for i in retrieved_chunks]
    # print(pairs)
    scores = model.predict(pairs)
    # print(scores)
    ranked = sorted(
        zip(retrieved_chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return ranked[:top_k]

# crossencoding(QUERY)

In [13]:
hits = 0
total = 0

with open("qa.json", "r", encoding="utf-8") as file:
    data = json.load(file)

for qa in data:
    question = qa["question"]
    gold_chunks = qa["gold_chunks"]

    if not gold_chunks:
        continue

    total += 1
    
    retrieved = crossencoding(query=question)
    retrieved_chunks = [chunk for chunk, score in retrieved]
    
    hit = any(
        any(
            gold_chunk["source"] == retrieved_chunk["source"]
            and gold_chunk["chunk_id"] == retrieved_chunk["chunk_id"]
            for retrieved_chunk in retrieved_chunks
        )
        for gold_chunk in gold_chunks
    )

    if hit:
        hits += 1

hits, total, hits / total if total else 0


(19, 21, 0.9047619047619048)

In [14]:

load_dotenv()
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [19]:
def build_prompt(question, retrieved_chunks):

    context = ""

    for chunk in retrieved_chunks:

        context += (
            f"Source: {chunk['source']}\n"
            f"{chunk['text']}\n\n"
        )

    prompt = f"""
            You are a helpful AI assistant.

            Answer Only using the context below.

            If the answer is not present in the context,
            reply exactly:

            "I don't know."

            Context
            --------------------
            {context}
            --------------------

            Question:
            {question}

            Answer:
            """

    return prompt

In [20]:
def generate_answer(prompt):

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    return response.choices[0].message.content


In [21]:
def rag_query(question):

    ranked_chunks = crossencoding(question)
    chunks = [chunk for chunk, score in ranked_chunks]

    prompt = build_prompt(question, chunks)

    answer = generate_answer(prompt)

    return {
        "question": question,
        "answer": answer,
        "sources": list(set(chunk["source"] for chunk in chunks)),
        "metadata": list(set((chunk["chunk_id"]) for chunk in chunks))
    }


## Generation Evaluation


In [15]:
def build_judge_prompt(question, gold_answer, generated_answer):

    return f"""
You are evaluating a RAG answer.

Classify the generated answer by comparing it with the gold answer.

Use only one of these labels:
- Correct
- Partial
- Incorrect

Return only valid JSON in this format:
{{
  "label": "Correct",
  "explanation": "Brief reason."
}}

Question:
{question}

Gold Answer:
{gold_answer}

Generated Answer:
{generated_answer}
""".strip()


In [16]:
def judge_answer(question, gold_answer, generated_answer):

    prompt = build_judge_prompt(
        question=question,
        gold_answer=gold_answer,
        generated_answer=generated_answer
    )

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )

    judgement = json.loads(response.choices[0].message.content)

    label = judgement.get("label", "Incorrect")
    if label not in {"Correct", "Partial", "Incorrect"}:
        label = "Incorrect"

    return {
        "label": label,
        "explanation": judgement.get("explanation", "")
    }


In [22]:
def evaluate_generation(qa_path="qa.json", output_path="generation_eval_results.json", max_questions=None):

    with open(qa_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    if max_questions is not None:
        data = data[:max_questions]

    results = []

    for idx, qa in enumerate(data, start=1):
        question = qa["question"]
        gold_answer = qa["gold_answer"]

        print(f"[{idx}/{len(data)}] {question}")

        rag_result = rag_query(question)
        generated_answer = rag_result["answer"]

        judgement = judge_answer(
            question=question,
            gold_answer=gold_answer,
            generated_answer=generated_answer
        )

        row = {
            "question": question,
            "gold_answer": gold_answer,
            "generated_answer": generated_answer,
            "label": judgement["label"],
            "explanation": judgement["explanation"],
            "sources": rag_result["sources"],
            "metadata": rag_result["metadata"]
        }

        results.append(row)

        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(results, file, indent=2, ensure_ascii=False)

        print(f"  {row['label']}: {row['explanation']}")

    summary = {
        "Correct": sum(row["label"] == "Correct" for row in results),
        "Partial": sum(row["label"] == "Partial" for row in results),
        "Incorrect": sum(row["label"] == "Incorrect" for row in results),
        "total": len(results)
    }

    return summary, results


In [23]:
generation_summary, generation_results = evaluate_generation(max_questions=5)
generation_summary


[1/5] What does VITON stand for?
  Correct: The generated answer matches the gold answer, differing only in case.
[2/5] Which deep learning architecture is mentioned for image generation in the try-on module?
  Correct: The generated answer 'U-net' matches the gold answer 'U-Net' with minor case difference.
[3/5] What are the two main modules of the proposed virtual try-on system?
  Correct: The generated answer correctly identifies the two main modules of the proposed virtual try-on system, albeit in a different order.
[4/5] What dataset is used for training and evaluation?
  Correct: The generated answer includes the gold answer and provides additional relevant information.
[5/5] Approximately how many image pairs are included in the VTON dataset?
  Correct: The generated answer matches the gold answer.


{'Correct': 5, 'Partial': 0, 'Incorrect': 0, 'total': 5}